In [1]:
!pip install pandas openpyxl mysql-connector-python

In [ ]:
import pandas as pd
import mysql.connector
from mysql.connector import Error
import warnings
warnings.filterwarnings('ignore')

EXCEL_FILE = r"C:\Users\bibha\Documents\Desktop\Project_2\Supply chain (3)\Supply chain\Dataset\Supply Chain-Project\Supply_Chain_Dataset.xlsx"

DB_CONFIG = {
    'host':     'localhost',
    'port':     3306,
    'user':     'root',          
    'password': 'your_password', 
}

DB_NAME = 'supply_chain_db'

print('Config ready ')

Config ready 


In [95]:
sheets = pd.read_excel(
    EXCEL_FILE,
    sheet_name=[
        'Dim_Product', 'Dim_Supplier', 'Dim_Warehouse',
        'Dim_Customer', 'Fact_Orders', 'Calender', 'Fact_Inventory'
    ]
)

dim_product    = sheets['Dim_Product'].copy()
dim_supplier   = sheets['Dim_Supplier'].copy()
dim_warehouse  = sheets['Dim_Warehouse'].copy()
dim_customer   = sheets['Dim_Customer'].copy()
fact_orders    = sheets['Fact_Orders'].copy()
dim_calendar   = sheets['Calender'].copy()
fact_inventory = sheets['Fact_Inventory'].copy()

print('All sheets loaded ')

All sheets loaded 


In [96]:
# Helper — strip leading/trailing whitespace from all string columns
def strip_strings(df):
    for col in df.columns:
        if df[col].dtype == 'object' or str(df[col].dtype) == 'string':
            df[col] = df[col].astype(str).str.strip()
    return df

# ── Dim_Product ──
dim_product = strip_strings(dim_product)
dim_product['Unit_Cost']  = dim_product['Unit_Cost'].round(2)
dim_product['Unit_Price'] = dim_product['Unit_Price'].round(2)
print(f"Dim_Product   : {dim_product.shape[0]} rows, {dim_product.shape[1]} cols — clean ")

# ── Dim_Supplier ──
dim_supplier = strip_strings(dim_supplier)
dim_supplier['Reliability_Score'] = dim_supplier['Reliability_Score'].round(2)
print(f"Dim_Supplier  : {dim_supplier.shape[0]} rows, {dim_supplier.shape[1]} cols — clean ")

# ── Dim_Warehouse ──
dim_warehouse = strip_strings(dim_warehouse)
print(f"Dim_Warehouse : {dim_warehouse.shape[0]} rows, {dim_warehouse.shape[1]} cols — clean ")

# ── Dim_Customer ──
dim_customer = strip_strings(dim_customer)
print(f"Dim_Customer  : {dim_customer.shape[0]} rows, {dim_customer.shape[1]} cols — clean ")

# ── Dim_Calendar ──
dim_calendar = dim_calendar.loc[:, ~dim_calendar.columns.str.startswith('Unnamed')]
dim_calendar = strip_strings(dim_calendar)
dim_calendar['Date'] = pd.to_datetime(dim_calendar['Date']).dt.date
# Rename reserved words
dim_calendar = dim_calendar.rename(columns={
    'Date':       'Cal_Date',
    'Year':       'Cal_Year',
    'Month':      'Cal_Month',
    'Year_Month': 'Cal_Year_Month',
    'Quarter':    'Cal_Quarter'
})
print(f"Dim_Calendar  : {dim_calendar.shape[0]} rows, {dim_calendar.shape[1]} cols — clean ")

# ── Fact_Orders ──
fact_orders = fact_orders.loc[:, ~fact_orders.columns.str.startswith('Unnamed')]
fact_orders = strip_strings(fact_orders)
for col in ['Order_Date', 'Ship_Date', 'Promised_Delivery_Date', 'Actual_Delivery_Date']:
    fact_orders[col] = pd.to_datetime(fact_orders[col]).dt.date
for col in ['Unit_Price', 'Unit_Cost', 'Revenue', 'COGS', 'Shipping_Cost', 'Fill_Rate_Pct']:
    fact_orders[col] = fact_orders[col].round(2)
print(f"Fact_Orders   : {fact_orders.shape[0]} rows, {fact_orders.shape[1]} cols — clean ")

# ── Fact_Inventory ──
fact_inventory = strip_strings(fact_inventory)
fact_inventory['Snapshot_Date']  = pd.to_datetime(fact_inventory['Snapshot_Date']).dt.date
fact_inventory['Days_Of_Supply'] = fact_inventory['Days_Of_Supply'].round(2)
print(f"Fact_Inventory: {fact_inventory.shape[0]} rows, {fact_inventory.shape[1]} cols — clean ")

print('\nColumn names in dim_calendar:', list(dim_calendar.columns))

Dim_Product   : 92 rows, 7 cols — clean 
Dim_Supplier  : 32 rows, 6 cols — clean 
Dim_Warehouse : 20 rows, 5 cols — clean 
Dim_Customer  : 1500 rows, 5 cols — clean 
Dim_Calendar  : 730 rows, 6 cols — clean 
Fact_Orders   : 12000 rows, 23 cols — clean 
Fact_Inventory: 6228 rows, 10 cols — clean 

Column names in dim_calendar: ['Cal_Date', 'Cal_Year', 'Cal_Month', 'Month_Name', 'Cal_Year_Month', 'Cal_Quarter']


In [99]:
tables = {
    'dim_product':    dim_product,
    'dim_supplier':   dim_supplier,
    'dim_warehouse':  dim_warehouse,
    'dim_customer':   dim_customer,
    'dim_calendar':   dim_calendar,
    'fact_orders':    fact_orders,
    'fact_inventory': fact_inventory
}

for name, df in tables.items():
    print(f"\n{'='*50}")
    print(f"  {name.upper()}")
    print(f"{'='*50}")
    
    # Shape
    print(f"  Rows : {df.shape[0]:,}")
    print(f"  Cols : {df.shape[1]}")
    
    # Duplicates
    dups = df.duplicated().sum()
    print(f"  Duplicates : {dups} {' FOUND' if dups > 0 else ' None'}")
    
    # Nulls
    nulls = df.isnull().sum()
    nulls = nulls[nulls > 0]
    if len(nulls) == 0:
        print(f"  Nulls      :  None")
    else:
        print(f"  Nulls      : ⚠ FOUND")
        for col, count in nulls.items():
            pct = (count / len(df)) * 100
            print(f"    → {col}: {count:,} ({pct:.1f}%)")
    
    # Negative values in numeric columns
    num_cols = df.select_dtypes(include='number').columns
    for col in num_cols:
        neg = (df[col] < 0).sum()
        if neg > 0:
            print(f"  Negatives  :   {col} has {neg} negative values")
    
    # Unique value counts for key categorical columns
    cat_cols = df.select_dtypes(include='object').columns
    if len(cat_cols) > 0:
        print(f"  Categoricals:")
        for col in cat_cols:
            print(f"    → {col}: {df[col].nunique()} unique values")


  DIM_PRODUCT
  Rows : 92
  Cols : 7
  Duplicates : 0  None
  Nulls      :  None
  Categoricals:
    → Product_ID: 92 unique values
    → Product_Name: 76 unique values
    → Category: 5 unique values
    → Sub_Category: 14 unique values
    → Primary_Supplier_ID: 27 unique values

  DIM_SUPPLIER
  Rows : 32
  Cols : 6
  Duplicates : 0  None
  Nulls      :  None
  Categoricals:
    → Supplier_ID: 32 unique values
    → Supplier_Name: 32 unique values
    → Supplier_Country: 7 unique values
    → Supplier_City: 13 unique values
    → Supplier_Tier: 3 unique values

  DIM_WAREHOUSE
  Rows : 20
  Cols : 5
  Duplicates : 0  None
  Nulls      :  None
  Categoricals:
    → Warehouse_ID: 20 unique values
    → Warehouse_City: 20 unique values
    → Warehouse_Country: 15 unique values
    → Warehouse_Region: 5 unique values

  DIM_CUSTOMER
  Rows : 1,500
  Cols : 5
  Duplicates : 0  None
  Nulls      :  None
  Categoricals:
    → Customer_ID: 1500 unique values
    → Customer_Region: 5 unique

In [101]:
dim_product.head(3)

,Product_ID,Product_Name,Category,Sub_Category,Unit_Cost,Unit_Price,Primary_Supplier_ID
0,PRD1000,Premium Smartphone Basic,Electronics,Smartphones,734.28,1170.60,SUP009
1,PRD1001,Premium Smartphone Pro,Electronics,Smartphones,692.28,1184.48,SUP032
2,PRD1002,Budget Smartphone Lite,Electronics,Smartphones,127.46,228.68,SUP008


In [103]:
fact_orders.head(3)

,Order_ID,Customer_ID,Product_ID,Supplier_ID,Warehouse_ID,Order_Date,Ship_Date,Promised_Delivery_Date,Actual_Delivery_Date,Ship_Mode,...,Unit_Price,Unit_Cost,Revenue,COGS,Shipping_Cost,Processing_Days,Transit_Days,Delay_Days,Delivery_Status,Fill_Rate_Pct
0,ORD100000,CUST00352,PRD1065,SUP001,WH14,2023-09-08,2023-09-10,2023-09-10,2023-09-11,Same-day,...,272.84,181.03,15279.04,10137.68,309.66,2,1,1,Slightly Delayed,100.0
1,ORD100001,CUST00336,PRD1019,SUP003,WH14,2024-07-20,2024-07-26,2024-07-27,2024-07-30,Air,...,12.85,8.14,77.10,48.84,18.93,6,4,3,Slightly Delayed,100.0
2,ORD100002,CUST00656,PRD1040,SUP015,WH17,2024-01-17,2024-01-19,2024-01-31,2024-01-27,Rail,...,35.11,25.69,70.22,51.38,2.38,2,8,0,On-Time,100.0


In [105]:
fact_inventory.head(3)

,Product_ID,Warehouse_ID,Snapshot_Date,Stock_On_Hand,Reorder_Level,Safety_Stock,Units_Received,Units_Shipped,Days_Of_Supply,Stockout_Flag
0,PRD1000,WH06,2024-01-31,1331,306,157,202,315,126.8,0
1,PRD1000,WH06,2024-02-29,1473,306,157,436,313,141.2,0
2,PRD1000,WH06,2024-03-31,1607,306,157,277,476,101.3,0


In [107]:
CREATE_DB_SQL = f"CREATE DATABASE IF NOT EXISTS `{DB_NAME}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci;"

TABLE_DDL = [
    """
    CREATE TABLE IF NOT EXISTS dim_product (
        `Product_ID`          VARCHAR(20)    PRIMARY KEY,
        `Product_Name`        VARCHAR(100)   NOT NULL,
        `Category`            VARCHAR(50)    NOT NULL,
        `Sub_Category`        VARCHAR(50)    NOT NULL,
        `Unit_Cost`           DECIMAL(10,2)  NOT NULL,
        `Unit_Price`          DECIMAL(10,2)  NOT NULL,
        `Primary_Supplier_ID` VARCHAR(20)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS dim_supplier (
        `Supplier_ID`        VARCHAR(20)   PRIMARY KEY,
        `Supplier_Name`      VARCHAR(100)  NOT NULL,
        `Supplier_Country`   VARCHAR(50)   NOT NULL,
        `Supplier_City`      VARCHAR(50)   NOT NULL,
        `Supplier_Tier`      CHAR(1)       NOT NULL,
        `Reliability_Score`  DECIMAL(5,2)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS dim_warehouse (
        `Warehouse_ID`       VARCHAR(10)  PRIMARY KEY,
        `Warehouse_City`     VARCHAR(50)  NOT NULL,
        `Warehouse_Country`  VARCHAR(50)  NOT NULL,
        `Warehouse_Region`   VARCHAR(50)  NOT NULL,
        `Capacity_Units`     INT          NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS dim_customer (
        `Customer_ID`       VARCHAR(20)  PRIMARY KEY,
        `Customer_Region`   VARCHAR(50)  NOT NULL,
        `Customer_Country`  VARCHAR(50)  NOT NULL,
        `Customer_City`     VARCHAR(50)  NOT NULL,
        `Customer_Segment`  VARCHAR(50)  NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS dim_calendar (
        `Cal_Date`        DATE        PRIMARY KEY,
        `Cal_Year`        SMALLINT    NOT NULL,
        `Cal_Month`       TINYINT     NOT NULL,
        `Month_Name`      VARCHAR(15) NOT NULL,
        `Cal_Year_Month`  VARCHAR(10) NOT NULL,
        `Cal_Quarter`     VARCHAR(5)  NOT NULL
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS fact_orders (
        `Order_ID`                VARCHAR(20)    PRIMARY KEY,
        `Customer_ID`             VARCHAR(20),
        `Product_ID`              VARCHAR(20),
        `Supplier_ID`             VARCHAR(20),
        `Warehouse_ID`            VARCHAR(10),
        `Order_Date`              DATE,
        `Ship_Date`               DATE,
        `Promised_Delivery_Date`  DATE,
        `Actual_Delivery_Date`    DATE,
        `Ship_Mode`               VARCHAR(20),
        `Carrier`                 VARCHAR(50),
        `Order_Quantity`          INT,
        `Shipped_Quantity`        INT,
        `Unit_Price`              DECIMAL(10,2),
        `Unit_Cost`               DECIMAL(10,2),
        `Revenue`                 DECIMAL(12,2),
        `COGS`                    DECIMAL(12,2),
        `Shipping_Cost`           DECIMAL(10,2),
        `Processing_Days`         TINYINT,
        `Transit_Days`            TINYINT,
        `Delay_Days`              TINYINT,
        `Delivery_Status`         VARCHAR(30),
        `Fill_Rate_Pct`           DECIMAL(6,2),
        FOREIGN KEY (`Customer_ID`)  REFERENCES dim_customer(`Customer_ID`),
        FOREIGN KEY (`Product_ID`)   REFERENCES dim_product(`Product_ID`),
        FOREIGN KEY (`Supplier_ID`)  REFERENCES dim_supplier(`Supplier_ID`),
        FOREIGN KEY (`Warehouse_ID`) REFERENCES dim_warehouse(`Warehouse_ID`)
    )
    """,
    """
    CREATE TABLE IF NOT EXISTS fact_inventory (
        `Product_ID`      VARCHAR(20),
        `Warehouse_ID`    VARCHAR(10),
        `Snapshot_Date`   DATE,
        `Stock_On_Hand`   INT,
        `Reorder_Level`   INT,
        `Safety_Stock`    INT,
        `Units_Received`  INT,
        `Units_Shipped`   INT,
        `Days_Of_Supply`  DECIMAL(8,2),
        `Stockout_Flag`   TINYINT(1),
        PRIMARY KEY (`Product_ID`, `Warehouse_ID`, `Snapshot_Date`),
        FOREIGN KEY (`Product_ID`)   REFERENCES dim_product(`Product_ID`),
        FOREIGN KEY (`Warehouse_ID`) REFERENCES dim_warehouse(`Warehouse_ID`)
    )
    """
]

print('DDL statements ready ')

DDL statements ready 


In [109]:
def load_dataframe(cursor, conn, df, table_name):
    cols         = ', '.join([f'`{c}`' for c in df.columns])
    placeholders = ', '.join(['%s'] * len(df.columns))
    updates      = ', '.join([f'`{c}`=VALUES(`{c}`)' for c in df.columns])
    sql = f"INSERT INTO `{table_name}` ({cols}) VALUES ({placeholders}) ON DUPLICATE KEY UPDATE {updates};"

    rows = [
        tuple(None if (val != val) else val for val in row)
        for row in df.itertuples(index=False, name=None)
    ]
    cursor.executemany(sql, rows)
    conn.commit()
    print(f'  Loaded {len(rows)} rows  →  {table_name}')


try:
    conn   = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()

    cursor.execute(CREATE_DB_SQL)
    cursor.execute(f'USE `{DB_NAME}`;')
    print(f'Database `{DB_NAME}` ready.')

    print('\nCreating tables...')
    for ddl in TABLE_DDL:
        cursor.execute(ddl)
    conn.commit()
    print('All tables created ')

    print('\nLoading data...')
    load_dataframe(cursor, conn, dim_product,    'dim_product')
    load_dataframe(cursor, conn, dim_supplier,   'dim_supplier')
    load_dataframe(cursor, conn, dim_warehouse,  'dim_warehouse')
    load_dataframe(cursor, conn, dim_customer,   'dim_customer')
    load_dataframe(cursor, conn, dim_calendar,   'dim_calendar')
    load_dataframe(cursor, conn, fact_orders,    'fact_orders')
    load_dataframe(cursor, conn, fact_inventory, 'fact_inventory')

    print('\n✅ All done! Data successfully loaded into MySQL.')

except Error as e:
    print(f'\n MySQL Error: {e}')

finally:
    if 'cursor' in locals(): cursor.close()
    if 'conn' in locals() and conn.is_connected():
        conn.close()
        print('Connection closed.')

Database `supply_chain_db` ready.

Creating tables...
All tables created 

Loading data...
  Loaded 92 rows  →  dim_product
  Loaded 32 rows  →  dim_supplier
  Loaded 20 rows  →  dim_warehouse
  Loaded 1500 rows  →  dim_customer
  Loaded 730 rows  →  dim_calendar
  Loaded 12000 rows  →  fact_orders
  Loaded 6228 rows  →  fact_inventory

✅ All done! Data successfully loaded into MySQL.
Connection closed.


## Step 8 — Quick Verification

In [113]:
try:
    conn   = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()
    
    cursor.execute(f'USE `{DB_NAME}`;')

    tables = ['dim_product','dim_supplier','dim_warehouse','dim_customer',
              'dim_calendar','fact_orders','fact_inventory']

    print(f"{'Table':<20} {'MySQL Rows':>12}")
    print('-' * 34)
    for t in tables:
        cursor.execute(f'SELECT COUNT(*) FROM `{t}`;')
        count = cursor.fetchone()[0]
        print(f"{t:<20} {count:>12,}")

except Error as e:
    print(f'error {e}')
finally:
    if 'cursor' in locals(): cursor.close()
    if 'conn' in locals() and conn.is_connected(): conn.close()

Table                  MySQL Rows
----------------------------------
dim_product                    92
dim_supplier                   32
dim_warehouse                  20
dim_customer                1,500
dim_calendar                  730
fact_orders                12,000
fact_inventory              6,228
